# Experiment B — MPJPE and normalized PCK vs optical-flow period

This corrected version keeps the Experiment B target metric definitions, but computes and aggregates them in the same manner as the previous Experiment A notebook.

Main alignment points:

- PCK normalization uses the same reference joints as Experiment A: `REF_JOINT_1 = 1`, `REF_JOINT_2 = 5`.
- PCK thresholds match Experiment A: `[0.1, 0.2, 0.4, 0.6]`.
- Dataset-level MPJPE is weighted by evaluated joint-point count: `sum_err_px / n_points`.
- Dataset-level PCK is computed by summing correct joint points, then dividing by total evaluated joint points.
- Per-joint MPJPE and PCK are weighted by sample count, as in Experiment A.
- The method label is parsed from the CSV filename instead of being hard-coded.

Prediction CSVs are treated as predictions only:

```text
timestamp, latency, joint0_x, joint0_y, ..., joint12_x, joint12_y
```

Ground truth is loaded from the original DHP19 skeleton logs and interpolated to the prediction timestamps.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 180)

try:
    from datasets.utils import constants as ds_constants, parsing as ds_parsing
    DATASET_UTILS_AVAILABLE = True
except Exception as exc:
    DATASET_UTILS_AVAILABLE = False
    DATASET_UTILS_IMPORT_ERROR = repr(exc)

# ---------------------------------------------------------------------------
# User configuration
# ---------------------------------------------------------------------------

# Leave None to auto-discover a folder containing results/raw/.
RESULTS_DIR = None

# Must match the --data_root used to run the experiment.
# DATA_ROOT = Path("/data/dhp19_testing_set_S13toS17")
DATA_ROOT = Path("/data/eh36m_testing_set_S9S11")

N_JOINTS = 13
# IMAGE_WIDTH_PX = 346
# IMAGE_HEIGHT_PX = 260

IMAGE_WIDTH_PX = 640
IMAGE_HEIGHT_PX = 280

# Normalized thresholds, not pixel thresholds.
# These are intentionally the same thresholds used by Experiment A.
# pck_0.2 means error <= 0.2 * reference_distance.
PCK_THRESHOLDS_NORM = [0.1, 0.2, 0.4, 0.6]
PRIMARY_PCK_THRESHOLD = 0.4

# Same PCK reference joints used in Experiment A.
# The notebook prints joint_names below so you can verify the indices.
REF_JOINT_1 = 1
REF_JOINT_2 = 5

# Keep this False for strict comparability with Experiment A.
# Experiment A computes MPJPE = err.mean() and PCK = pck_binary.mean() * 100
# without masking GT joints.
USE_VALID_GT_MASK_FOR_METRICS = False
IGNORE_GT_ZERO_ZERO = True
IGNORE_GT_OUTSIDE_IMAGE = True

# Use this only if GT and prediction timestamps have a constant offset.
PRED_TIMESTAMP_OFFSET_S = 0.0

# Leave None to infer camera-specific GT from the CSV name.
# Example: S13_1_1__ch2dvs -> S13_1_1/ch2GT200Hzskeleton/data.log
FORCE_GT_FOLDER = None

ANALYSIS_OUT_DIR = Path("analysis_outputs_expB")
ANALYSIS_OUT_DIR.mkdir(parents=True, exist_ok=True)

if DATASET_UTILS_AVAILABLE:
    joint_names = list(ds_constants.HPECoreSkeleton.KEYPOINTS_MAP.keys())
else:
    joint_names = [f"joint{i}" for i in range(N_JOINTS)]

if len(joint_names) < N_JOINTS:
    joint_names += [f"joint{i}" for i in range(len(joint_names), N_JOINTS)]
joint_names = joint_names[:N_JOINTS]

print("DATASET_UTILS_AVAILABLE:", DATASET_UTILS_AVAILABLE)
if not DATASET_UTILS_AVAILABLE:
    print("Import error:", DATASET_UTILS_IMPORT_ERROR)
    print("Run this notebook in the same environment as the old notebook, or add the project repo to PYTHONPATH.")

print("DATA_ROOT:", DATA_ROOT)
print("\nJoint indices:")
for i, name in enumerate(joint_names):
    tag = ""
    if i == REF_JOINT_1:
        tag = "  <-- REF_JOINT_1"
    if i == REF_JOINT_2:
        tag = "  <-- REF_JOINT_2"
    print(f"  {i:2d}: {name}{tag}")

print(f"\nNormalized PCK reference: {REF_JOINT_1} ({joint_names[REF_JOINT_1]}) -> {REF_JOINT_2} ({joint_names[REF_JOINT_2]})")
print("PCK thresholds:", PCK_THRESHOLDS_NORM)


DATASET_UTILS_AVAILABLE: True
DATA_ROOT: /data/eh36m_testing_set_S9S11

Joint indices:
   0: head
   1: shoulder_right  <-- REF_JOINT_1
   2: shoulder_left
   3: elbow_right
   4: elbow_left
   5: hip_left  <-- REF_JOINT_2
   6: hip_right
   7: wrist_right
   8: wrist_left
   9: knee_right
  10: knee_left
  11: ankle_right
  12: ankle_left

Normalized PCK reference: 1 (shoulder_right) -> 5 (hip_left)
PCK thresholds: [0.1, 0.2, 0.4, 0.6]


## 1. Find prediction CSVs


In [ ]:
def decode_period_token(token):
    return float(str(token).replace("p", "."))

def find_results_dir():
    if RESULTS_DIR is not None:
        p = Path(RESULTS_DIR).expanduser().resolve()
        if not (p / "raw").exists():
            raise FileNotFoundError(f"RESULTS_DIR exists but does not contain raw/: {p}")
        return p

    candidates = [
        Path.cwd(),
        Path.cwd() / "results",
        Path.cwd() / "dhp19_full_test" / "results",
        Path.cwd() / "eh36m_full_test" / "results",
        Path.cwd().parent / "results",
        Path.cwd().parent / "dhp19_full_test" / "results",
        Path.cwd().parent / "eh36m_full_test" / "results",
    ]
    for p in candidates:
        if (p / "raw").exists():
            return p.resolve()

    for p in [Path.cwd(), Path.cwd().parent]:
        matches = list(p.glob("*/results/raw"))
        if matches:
            return matches[0].parent.resolve()

    raise FileNotFoundError("Could not find results/raw/. Set RESULTS_DIR manually.")

RESULTS = find_results_dir()
RAW_DIR = RESULTS / "raw"
LOG_DIR = RESULTS / "logs"

csv_files = sorted(RAW_DIR.glob("net_*/fp_*/*.csv"))

print("RESULTS:", RESULTS)
print("RAW_DIR:", RAW_DIR)
print("CSV files:", len(csv_files))
for p in csv_files[:5]:
    print("  ", p.relative_to(RESULTS))

if not csv_files:
    raise RuntimeError("No CSV files found under raw/net_*/fp_*/*.csv")


RESULTS: /eh36m_full_test/results
RAW_DIR: /eh36m_full_test/results/raw
CSV files: 0


RuntimeError: No CSV files found under raw/net_*/fp_*/*.csv

## 2. Loaders and metadata helpers


In [3]:
def parse_config_from_path(csv_path):
    rel = csv_path.relative_to(RAW_DIR)
    net_token = rel.parts[0].replace("net_", "")
    fp_token = rel.parts[1].replace("fp_", "")
    network_period_s = decode_period_token(net_token)
    flow_period_s = decode_period_token(fp_token)
    return {
        "flow_period_s": flow_period_s,
        "flow_rate_hz": 1.0 / flow_period_s,
        "network_period_s": network_period_s,
        "detection_rate_hz": 1.0 / network_period_s,
        "net_token": net_token,
        "fp_token": fp_token,
    }

def method_from_csv(csv_path):
    """
    Parse the method label from the CSV filename.

    Experiment A distinguishes MoveEnet-only predictions from MoveEnet+OFK.
    Experiment B is often OFK-only, but this parser keeps the grouping correct
    if both methods are present in the same raw folder.
    """
    name = csv_path.stem.lower()

    if "moveenet_only" in name or "movenet_only" in name:
        return "MoveEnet only"

    if "moveenet_ofk" in name or "movenet_ofk" in name or "ofk" in name:
        return "MoveEnet + OFK"

    # Experiment B files are usually OFK outputs. Keep this explicit fallback so
    # files without a method token are still processed.
    return "MoveEnet + OFK"


def sequence_key_from_csv(csv_path):
    """
    Remove the method/configuration suffix from a prediction filename and return
    the sequence key used to locate the GT skeleton file.

    Examples:
        S13_1_1__ch2dvs__moveenet_ofk_np_0p1_fp_0p01.csv -> S13_1_1__ch2dvs
        S13_1_1__ch2dvs__movenet_only.csv               -> S13_1_1__ch2dvs
    """
    name = csv_path.stem

    # Remove method-specific suffixes first.
    name = re.sub(r"__(?:moveenet|movenet)_(?:ofk|only).*$", "", name, flags=re.IGNORECASE)

    # Fallback for any remaining OFK-style suffix.
    name = re.sub(r"__.*ofk.*$", "", name, flags=re.IGNORECASE)

    return name


def split_sequence_and_camera(seq_key):
    parts = seq_key.split("__")
    camera_folder = None
    seq_parts = parts
    if parts and re.match(r"^ch\d+dvs$", parts[-1]):
        camera_folder = parts[-1]
        seq_parts = parts[:-1]
    if not seq_parts:
        seq_parts = [seq_key]

    camera_id = None
    if camera_folder:
        m = re.match(r"^(ch\d+)dvs$", camera_folder)
        if m:
            camera_id = m.group(1)

    return seq_parts, camera_folder, camera_id

def gt_candidates_from_seq_key(seq_key):
    seq_parts, camera_folder, camera_id = split_sequence_and_camera(seq_key)
    seq_dir = DATA_ROOT.joinpath(*seq_parts)

    candidates = []
    if FORCE_GT_FOLDER is not None:
        candidates.append(seq_dir / FORCE_GT_FOLDER / "data.log")
    if camera_id is not None:
        candidates.append(seq_dir / f"{camera_id}GT200Hzskeleton" / "data.log")
    if camera_folder is not None:
        candidates.append(seq_dir / camera_folder.replace("dvs", "GT200Hzskeleton") / "data.log")

    candidates.append(seq_dir / "ch0GT200Hzskeleton" / "data.log")
    candidates.append(seq_dir / "GT200Hzskeleton" / "data.log")

    # Old-style fallback where "__" encoded folder separators.
    rel_old = DATA_ROOT.joinpath(*seq_key.split("__"))
    candidates.append(rel_old / "ch0GT200Hzskeleton" / "data.log")

    unique = []
    seen = set()
    for c in candidates:
        c = Path(c)
        if c not in seen:
            unique.append(c)
            seen.add(c)
    return unique

def find_gt_path(seq_key):
    candidates = gt_candidates_from_seq_key(seq_key)
    for p in candidates:
        if p.exists():
            return p, candidates
    return None, candidates

def load_prediction_csv(csv_path):
    df = pd.read_csv(csv_path)
    if "timestamp" not in df.columns:
        raise ValueError(f"Missing timestamp column in {csv_path}")

    ts = pd.to_numeric(df["timestamp"], errors="coerce").to_numpy(dtype=float) + PRED_TIMESTAMP_OFFSET_S
    joints = np.full((len(df), N_JOINTS, 2), np.nan, dtype=np.float64)

    for j in range(N_JOINTS):
        x_candidates = [f"joint{j}_x", f"pred_joint{j}_x", f"pred_j{j}_x", f"prediction_joint{j}_x"]
        y_candidates = [f"joint{j}_y", f"pred_joint{j}_y", f"pred_j{j}_y", f"prediction_joint{j}_y"]
        x_col = next((c for c in x_candidates if c in df.columns), None)
        y_col = next((c for c in y_candidates if c in df.columns), None)
        if x_col is None or y_col is None:
            raise ValueError(f"Missing prediction columns for joint {j} in {csv_path}")
        joints[:, j, 0] = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
        joints[:, j, 1] = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)

    return df, ts, joints

def load_gt_data(gt_path):
    if not DATASET_UTILS_AVAILABLE:
        raise RuntimeError("datasets.utils could not be imported. Run this notebook in the same environment as the old notebook.")
    return ds_parsing.import_yarp_skeleton_data(gt_path, multi_channel=False)

def interp_gt_to_pred_ts(gt_data, ts_pred):
    ts_gt_raw = np.asarray(gt_data["ts"], dtype=np.float64)
    if len(ts_gt_raw) == 0:
        raise ValueError("GT file contains no timestamps.")

    ts_gt = np.concatenate(([0.0], ts_gt_raw, [ts_gt_raw[-1] + 1.0]))
    joints_gt = np.zeros((len(ts_pred), N_JOINTS, 2), dtype=np.float64)

    for j, joint in enumerate(joint_names):
        if joint not in gt_data:
            raise KeyError(f"GT joint {joint!r} not found. Available keys: {list(gt_data.keys())[:20]}")
        xy_raw = np.asarray(gt_data[joint], dtype=np.float64)
        x = np.concatenate(([xy_raw[0, 0]], xy_raw[:, 0], [xy_raw[-1, 0]]))
        y = np.concatenate(([xy_raw[0, 1]], xy_raw[:, 1], [xy_raw[-1, 1]]))
        joints_gt[:, j, 0] = np.interp(ts_pred, ts_gt, x)
        joints_gt[:, j, 1] = np.interp(ts_pred, ts_gt, y)

    return joints_gt

def valid_gt_mask(joints_gt):
    ok = np.isfinite(joints_gt[:, :, 0]) & np.isfinite(joints_gt[:, :, 1])
    if IGNORE_GT_ZERO_ZERO:
        ok &= ~((joints_gt[:, :, 0] == 0) & (joints_gt[:, :, 1] == 0))
    if IGNORE_GT_OUTSIDE_IMAGE:
        ok &= (
            (joints_gt[:, :, 0] >= 0) & (joints_gt[:, :, 0] <= IMAGE_WIDTH_PX) &
            (joints_gt[:, :, 1] >= 0) & (joints_gt[:, :, 1] <= IMAGE_HEIGHT_PX)
        )
    return ok


## 3. Normalized PCK implementation

This is the main correction. It uses normalized thresholds, not fixed pixel thresholds.


In [4]:
def calculate_pck_binary(joints_pred: np.ndarray, joints_gt: np.ndarray, threshold: float) -> np.ndarray:
    err_dist = np.linalg.norm(joints_pred - joints_gt, axis=2)

    ref_vec = joints_gt[:, REF_JOINT_2, :] - joints_gt[:, REF_JOINT_1, :]
    ref_dist = np.linalg.norm(ref_vec, axis=1)
    ref_dist = np.maximum(ref_dist, 1e-6)

    return (err_dist <= threshold * ref_dist[:, np.newaxis]).astype(np.float32)

def calculate_pck_binary_masked(joints_pred: np.ndarray, joints_gt: np.ndarray, threshold: float, mask: np.ndarray) -> np.ndarray:
    pck = calculate_pck_binary(joints_pred, joints_gt, threshold).astype(np.float64)
    pck[~mask] = np.nan
    return pck

def compute_sequence_metrics(joints_pred, joints_gt):
    if USE_VALID_GT_MASK_FOR_METRICS:
        raise ValueError(
            "USE_VALID_GT_MASK_FOR_METRICS must be False to match Experiment A. "
            "Experiment A does not mask GT joints during MPJPE/PCK computation."
        )

    err = np.linalg.norm(joints_pred - joints_gt, axis=2)
    valid = valid_gt_mask(joints_gt)

    if USE_VALID_GT_MASK_FOR_METRICS:
        err_for_metrics = err.copy()
        err_for_metrics[~valid] = np.nan
        n_points = int(np.isfinite(err_for_metrics).sum())
        sum_err_px = float(np.nansum(err_for_metrics))
        mpjpe_px = sum_err_px / n_points if n_points else np.nan
        per_joint_err = np.nanmean(err_for_metrics, axis=0)
    else:
        # Exact style from your snippet: err.mean(), err.sum(), err.size.
        err_for_metrics = err
        n_points = int(err.size)
        sum_err_px = float(err.sum())
        mpjpe_px = float(err.mean())
        per_joint_err = err.mean(axis=0)

    row = {
        "n_samples": int(err.shape[0]),
        "n_frames": int(err.shape[0]),
        "sum_err_px": sum_err_px,
        "n_points": n_points,
        "mpjpe_px": mpjpe_px,
    }

    for i, joint in enumerate(joint_names):
        row[f"mpjpe_{joint}"] = float(per_joint_err[i])
        if USE_VALID_GT_MASK_FOR_METRICS:
            row[f"sum_err_{joint}"] = float(np.nansum(err_for_metrics[:, i]))
            row[f"n_points_{joint}"] = int(np.isfinite(err_for_metrics[:, i]).sum())
        else:
            row[f"sum_err_{joint}"] = float(err[:, i].sum())
            row[f"n_points_{joint}"] = int(err.shape[0])

    for th in PCK_THRESHOLDS_NORM:
        if USE_VALID_GT_MASK_FOR_METRICS:
            pck_binary = calculate_pck_binary_masked(joints_pred, joints_gt, th, valid)
            row[f"pck_{th:.1f}"] = float(np.nanmean(pck_binary) * 100.0)
            row[f"correct_pck_{th:.1f}"] = float(np.nansum(pck_binary))
            row[f"n_pck_points_{th:.1f}"] = int(np.isfinite(pck_binary).sum())
            for j, joint in enumerate(joint_names):
                row[f"pck_{th:.1f}_{joint}"] = float(np.nanmean(pck_binary[:, j]) * 100.0)
                row[f"correct_pck_{th:.1f}_{joint}"] = float(np.nansum(pck_binary[:, j]))
        else:
            # Exact style from your snippet: pck_binary.mean() * 100.0.
            pck_binary = calculate_pck_binary(joints_pred, joints_gt, th)
            row[f"pck_{th:.1f}"] = float(pck_binary.mean() * 100.0)
            row[f"correct_pck_{th:.1f}"] = float(pck_binary.sum())
            row[f"n_pck_points_{th:.1f}"] = int(pck_binary.size)
            for j, joint in enumerate(joint_names):
                row[f"pck_{th:.1f}_{joint}"] = float(pck_binary[:, j].mean() * 100.0)
                row[f"correct_pck_{th:.1f}_{joint}"] = float(pck_binary[:, j].sum())

    return row


## 4. Compute per-sequence metrics


In [5]:
gt_cache = {}
rows = []
missing_gt_rows = []
failed_rows = []

for csv_path in csv_files:
    cfg = parse_config_from_path(csv_path)
    method = method_from_csv(csv_path)
    seq_key = sequence_key_from_csv(csv_path)
    seq_parts, camera_folder, camera_id = split_sequence_and_camera(seq_key)

    gt_path, candidates = find_gt_path(seq_key)
    if gt_path is None:
        missing_gt_rows.append({
            "csv_path": str(csv_path),
            "csv_file": str(csv_path.relative_to(RESULTS)),
            "sequence": seq_key,
            "camera_folder": camera_folder,
            "all_candidates": " | ".join(str(c) for c in candidates),
        })
        continue

    try:
        _, ts_pred, joints_pred = load_prediction_csv(csv_path)

        if gt_path not in gt_cache:
            gt_cache[gt_path] = load_gt_data(gt_path)
        gt_data = gt_cache[gt_path]

        joints_gt = interp_gt_to_pred_ts(gt_data, ts_pred)
        metrics = compute_sequence_metrics(joints_pred, joints_gt)

        duration_s = float(np.nanmax(ts_pred) - np.nanmin(ts_pred)) if len(ts_pred) else np.nan

        row = {
            "csv_path": str(csv_path),
            "csv_file": str(csv_path.relative_to(RESULTS)),
            "sequence": seq_key,
            "method": method,
            "gt_file": str(gt_path),
            "sequence_path": "/".join(seq_parts),
            "camera_folder": camera_folder,
            "camera_id": camera_id,
            "duration_s": duration_s,
            "ref_joint_1": REF_JOINT_1,
            "ref_joint_2": REF_JOINT_2,
            "ref_joint_1_name": joint_names[REF_JOINT_1],
            "ref_joint_2_name": joint_names[REF_JOINT_2],
            **cfg,
            **metrics,
        }
        rows.append(row)

    except Exception as exc:
        failed_rows.append({
            "csv_path": str(csv_path),
            "csv_file": str(csv_path.relative_to(RESULTS)),
            "sequence": seq_key,
            "gt_file": str(gt_path),
            "error": repr(exc),
        })

df_seq = pd.DataFrame(rows)
df_missing_gt = pd.DataFrame(missing_gt_rows)
df_failed = pd.DataFrame(failed_rows)

if not df_seq.empty:
    df_seq = df_seq.sort_values(["network_period_s", "flow_period_s", "sequence"]).reset_index(drop=True)

print(f"Processed successfully : {len(df_seq)} / {len(csv_files)}")
print(f"Missing GT files       : {len(df_missing_gt)}")
print(f"Failed files           : {len(df_failed)}")
print(f"GT files loaded        : {len(gt_cache)}")

if len(df_missing_gt):
    print("\nExample missing GT rows:")
    display(df_missing_gt.head(10))

if len(df_failed):
    print("\nExample failed rows:")
    display(df_failed.head(10))

if df_seq.empty:
    raise RuntimeError("No metrics were computed. Check DATA_ROOT, GT folder naming, and datasets.utils availability.")

visible_cols = [
    "sequence", "method", "flow_period_s", "flow_rate_hz", "network_period_s", "detection_rate_hz",
    "n_samples", "mpjpe_px",
] + [f"pck_{th:.1f}" for th in PCK_THRESHOLDS_NORM]

display(df_seq[visible_cols].head())


Processed successfully : 0 / 720
Missing GT files       : 720
Failed files           : 0
GT files loaded        : 0

Example missing GT rows:


,csv_path,csv_file,sequence,camera_folder,all_candidates
0,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_1__ch2dvs__moveene...,S13_1_1__ch2dvs,ch2dvs,/data/eh36m_testing_set_S9S11/S13_1_1/ch2GT200...
1,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_1__ch3dvs__moveene...,S13_1_1__ch3dvs,ch3dvs,/data/eh36m_testing_set_S9S11/S13_1_1/ch3GT200...
2,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_2__ch2dvs__moveene...,S13_1_2__ch2dvs,ch2dvs,/data/eh36m_testing_set_S9S11/S13_1_2/ch2GT200...
3,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_2__ch3dvs__moveene...,S13_1_2__ch3dvs,ch3dvs,/data/eh36m_testing_set_S9S11/S13_1_2/ch3GT200...
4,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_3__ch2dvs__moveene...,S13_1_3__ch2dvs,ch2dvs,/data/eh36m_testing_set_S9S11/S13_1_3/ch2GT200...
5,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_3__ch3dvs__moveene...,S13_1_3__ch3dvs,ch3dvs,/data/eh36m_testing_set_S9S11/S13_1_3/ch3GT200...
6,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_4__ch2dvs__moveene...,S13_1_4__ch2dvs,ch2dvs,/data/eh36m_testing_set_S9S11/S13_1_4/ch2GT200...
7,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_4__ch3dvs__moveene...,S13_1_4__ch3dvs,ch3dvs,/data/eh36m_testing_set_S9S11/S13_1_4/ch3GT200...
8,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_5__ch2dvs__moveene...,S13_1_5__ch2dvs,ch2dvs,/data/eh36m_testing_set_S9S11/S13_1_5/ch2GT200...
9,/workspace/moveEnetFlow/experiments/expB_accur...,raw/net_0p02/fp_0p005/S13_1_5__ch3dvs__moveene...,S13_1_5__ch3dvs,ch3dvs,/data/eh36m_testing_set_S9S11/S13_1_5/ch3GT200...


RuntimeError: No metrics were computed. Check DATA_ROOT, GT folder naming, and datasets.utils availability.

## 5. Dataset-level weighted aggregation

This keeps aggregation exact:

- `mpjpe_px = sum_err_px / n_points`
- `pck_0.2 = sum(correct_pck_0.2) / sum(n_pck_points_0.2) * 100`


In [ ]:
if df_seq.empty:
    raise RuntimeError("No valid sequence rows to aggregate.")

group_cols = ["flow_period_s", "flow_rate_hz", "network_period_s", "detection_rate_hz", "method"]

# -----------------------------------------------------------------------------
# Dataset-level weighted aggregation, aligned with Experiment A
# -----------------------------------------------------------------------------
# MPJPE:
#   sum all joint errors over all rows in the group, then divide by total
#   evaluated joint points.
#
# PCK:
#   sum all correct joint points over all rows in the group, then divide by total
#   evaluated joint points.
#
# This avoids unweighted averages across sequences and matches the aggregation
# logic in Experiment A.
# -----------------------------------------------------------------------------

pck_cols = [
    f"pck_{th:.1f}"
    for th in PCK_THRESHOLDS_NORM
    if f"pck_{th:.1f}" in df_seq.columns
]

work = df_seq.copy()

# Keep explicit correct-point columns. If an older run does not contain them,
# reconstruct them from the per-sequence percentages and denominators.
for th in PCK_THRESHOLDS_NORM:
    pck_col = f"pck_{th:.1f}"
    correct_col = f"correct_pck_{th:.1f}"
    n_col = f"n_pck_points_{th:.1f}"

    if pck_col not in work.columns:
        continue

    if correct_col not in work.columns:
        work[correct_col] = (work[pck_col] / 100.0) * work["n_points"]

    if n_col not in work.columns:
        work[n_col] = work["n_points"]

agg_dict = {
    "n_csv_files": ("csv_path", "count"),
    "n_sequences": ("sequence", "nunique"),
    "n_samples": ("n_samples", "sum"),
    "n_frames": ("n_frames", "sum"),
    "sum_err_px": ("sum_err_px", "sum"),
    "n_points": ("n_points", "sum"),
    "evaluated_duration_s": ("duration_s", "sum"),
}

for th in PCK_THRESHOLDS_NORM:
    pck_col = f"pck_{th:.1f}"
    correct_col = f"correct_pck_{th:.1f}"
    n_col = f"n_pck_points_{th:.1f}"

    if pck_col in work.columns:
        agg_dict[correct_col] = (correct_col, "sum")
        agg_dict[n_col] = (n_col, "sum")

for joint in joint_names:
    sum_col = f"sum_err_{joint}"
    n_joint_col = f"n_points_{joint}"

    if sum_col in work.columns:
        agg_dict[sum_col] = (sum_col, "sum")
    if n_joint_col in work.columns:
        agg_dict[n_joint_col] = (n_joint_col, "sum")

    for th in PCK_THRESHOLDS_NORM:
        correct_joint_col = f"correct_pck_{th:.1f}_{joint}"
        if correct_joint_col in work.columns:
            agg_dict[correct_joint_col] = (correct_joint_col, "sum")

summary = (
    work
    .groupby(group_cols, as_index=False)
    .agg(**agg_dict)
)

summary["mpjpe_px"] = summary["sum_err_px"] / summary["n_points"]

for th in PCK_THRESHOLDS_NORM:
    pck_col = f"pck_{th:.1f}"
    correct_col = f"correct_pck_{th:.1f}"
    n_col = f"n_pck_points_{th:.1f}"

    if correct_col in summary.columns and n_col in summary.columns:
        summary[pck_col] = summary[correct_col] / summary[n_col] * 100.0

# Per-joint metrics are weighted by the number of evaluated samples, as in
# Experiment A. With strict Experiment-A mode, n_points_joint == n_samples.
for joint in joint_names:
    sum_col = f"sum_err_{joint}"
    if sum_col in summary.columns:
        summary[f"mpjpe_{joint}"] = summary[sum_col] / summary["n_samples"]

    for th in PCK_THRESHOLDS_NORM:
        pck_joint_col = f"pck_{th:.1f}_{joint}"
        correct_joint_col = f"correct_pck_{th:.1f}_{joint}"

        if correct_joint_col in summary.columns:
            summary[pck_joint_col] = summary[correct_joint_col] / summary["n_samples"] * 100.0

summary["flow_to_detection_rate_ratio"] = summary["flow_rate_hz"] / summary["detection_rate_hz"]
summary["flow_steps_per_detection"] = summary["network_period_s"] / summary["flow_period_s"]

summary = summary.sort_values(["network_period_s", "flow_period_s", "method"]).reset_index(drop=True)

display_cols = [
    "flow_period_s", "flow_rate_hz", "network_period_s", "detection_rate_hz",
    "method", "n_csv_files", "n_sequences", "n_samples", "n_points", "mpjpe_px",
] + [col for col in pck_cols if col in summary.columns]

display(summary[display_cols])

summary_path = ANALYSIS_OUT_DIR / "experimentB_summary_by_period.csv"
seq_path = ANALYSIS_OUT_DIR / "experimentB_metrics_by_sequence.csv"
missing_path = ANALYSIS_OUT_DIR / "experimentB_missing_gt.csv"
failed_path = ANALYSIS_OUT_DIR / "experimentB_failed_files.csv"

summary.to_csv(summary_path, index=False)
df_seq.to_csv(seq_path, index=False)
df_missing_gt.to_csv(missing_path, index=False)
df_failed.to_csv(failed_path, index=False)

print("Saved:", summary_path)
print("Saved:", seq_path)
print("Saved:", missing_path)
print("Saved:", failed_path)


## Consistency with Experiment A

The table above is the Experiment-B equivalent of Experiment A's `combined_dataset_weighted_metrics.csv`: it is grouped by period and method, and the dataset metrics are weighted by joint/sample counts rather than by unweighted sequence averages.


## 6. Plots


In [ ]:
FIG_DIR = ANALYSIS_OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved:", path)

def add_period_secondary_axis(ax):
    def hz_to_s(hz):
        hz = np.asarray(hz, dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            return 1.0 / hz
    def s_to_hz(s):
        s = np.asarray(s, dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            return 1.0 / s
    secax = ax.secondary_xaxis("top", functions=(hz_to_s, s_to_hz))
    secax.set_xlabel("Optical-flow period (s)")
    return secax

def plot_vs_flow(y_col, y_label, title, filename, ylim=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    for method in sorted(summary["method"].dropna().unique()):
        for net_period, sub in summary[summary["method"] == method].groupby("network_period_s"):
            sub = sub.sort_values("flow_rate_hz")
            ax.plot(
                sub["flow_rate_hz"],
                sub[y_col],
                marker="o",
                label=f"{method}, detection {1/net_period:g} Hz ({net_period:g} s)",
            )

    ax.set_xlabel("Optical-flow rate (Hz)")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    savefig(filename)
    plt.show()

plot_vs_flow(
    "mpjpe_px",
    "Dataset MPJPE (pixels)",
    "Experiment B: dataset MPJPE vs optical-flow rate",
    "mpjpe_vs_flow_rate_dataset_weighted.png",
)

for th in PCK_THRESHOLDS_NORM:
    pck_col = f"pck_{th:.1f}"
    if pck_col not in summary.columns:
        continue

    plot_vs_flow(
        pck_col,
        f"Dataset PCK@{th:.1f} (%)",
        f"Experiment B: dataset PCK@{th:.1f} vs optical-flow rate",
        f"pck_{th:.1f}_vs_flow_rate_dataset_weighted.png",
        ylim=(40, 105),
    )


In [ ]:
def heatmap(value_col, title, label, filename, fmt=".2f"):
    pivot = summary.pivot_table(
        index="detection_rate_hz",
        columns="flow_rate_hz",
        values=value_col,
        aggfunc="mean",
    ).sort_index(ascending=False).sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.to_numpy(), aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{x:g}" for x in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{x:g}" for x in pivot.index])
    ax.set_xlabel("Optical-flow rate (Hz)")
    ax.set_ylabel("Network detection rate (Hz)")
    ax.set_title(title)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.iloc[i, j]
            if pd.notna(v):
                ax.text(j, i, format(v, fmt), ha="center", va="center")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(label)
    savefig(filename)
    plt.show()
    return pivot

mpjpe_pivot = heatmap(
    "mpjpe_px",
    "MPJPE heatmap",
    "2D MPJPE (pixels)",
    "mpjpe_heatmap.png",
    ".2f",
)

pck_pivot = heatmap(
    f"pck_{PRIMARY_PCK_THRESHOLD:.1f}",
    f"Normalized PCK@{PRIMARY_PCK_THRESHOLD:.1f} heatmap",
    f"Normalized PCK@{PRIMARY_PCK_THRESHOLD:.1f} (%)",
    "normalized_pck_heatmap.png",
    ".1f",
)


## 7. Per-joint summary


In [ ]:
joint_rows = []

for _, r in summary.iterrows():
    for joint in joint_names:
        row = {
            "flow_period_s": r["flow_period_s"],
            "flow_rate_hz": r["flow_rate_hz"],
            "network_period_s": r["network_period_s"],
            "detection_rate_hz": r["detection_rate_hz"],
            "joint": joint,
            "mpjpe_px": r[f"mpjpe_{joint}"],
        }
        for th in PCK_THRESHOLDS_NORM:
            row[f"pck_{th:.1f}"] = r[f"pck_{th:.1f}_{joint}"]
        joint_rows.append(row)

df_joint_summary = pd.DataFrame(joint_rows)
display(df_joint_summary.head(30))

joint_summary_path = ANALYSIS_OUT_DIR / "experimentB_per_joint_summary.csv"
df_joint_summary.to_csv(joint_summary_path, index=False)
print("Saved:", joint_summary_path)

for flow_rate, sub in df_joint_summary.groupby("flow_rate_hz"):
    d = sub.groupby("joint", as_index=False).agg(mpjpe_px=("mpjpe_px", "mean"))
    d["joint"] = pd.Categorical(d["joint"], categories=joint_names, ordered=True)
    d = d.sort_values("joint")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(d["joint"].astype(str), d["mpjpe_px"])
    ax.set_xlabel("Joint")
    ax.set_ylabel("2D MPJPE (pixels)")
    ax.set_title(f"Per-joint MPJPE at optical-flow rate {flow_rate:g} Hz")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.3)
    savefig(f"per_joint_mpjpe_flow_{flow_rate:g}Hz.png".replace(".", "p"))
    plt.show()


# 8. Specific desired images

In [ ]:
# Plot normalized PCK@0.2 with adjustable y-axis zoom

PCK_THRESHOLD = 0.2
PCK_COL = f"pck_{PCK_THRESHOLD:.1f}"

YLIM_PCK02 = (85, 90)

fig, ax = plt.subplots(figsize=(8, 5))

for net_period, sub in summary.groupby("network_period_s"):
    sub = sub.sort_values("flow_rate_hz")

    ax.plot(
        sub["flow_rate_hz"],
        sub[PCK_COL],
        marker="o",
        linewidth=2,
        label=f"detection {1 / net_period:g} Hz ({net_period:g} s)",
    )

ax.set_xlabel("Optical-flow rate (Hz)")
ax.set_ylabel(f"Normalized PCK@{PCK_THRESHOLD:.1f} (%)")
ax.set_title(f"Experiment B: normalized PCK@{PCK_THRESHOLD:.1f} vs optical-flow rate")
ax.grid(True, alpha=0.3)
ax.legend()

if YLIM_PCK02 is not None:
    ax.set_ylim(YLIM_PCK02)


savefig(f"normalized_pck_{PCK_THRESHOLD:.1f}_vs_flow_rate_zoom.png")
plt.show()

In [ ]:
# Plot normalized PCK@0.4 with adjustable y-axis zoom

PCK_THRESHOLD = 0.4
PCK_COL = f"pck_{PCK_THRESHOLD:.1f}"

# Modify this to zoom in.
# Example: (65, 80)
# Use None to let matplotlib choose automatically.
# YLIM_PCK04 = None
YLIM_PCK04 = (97, 99)

fig, ax = plt.subplots(figsize=(8, 5))

for net_period, sub in summary.groupby("network_period_s"):
    sub = sub.sort_values("flow_rate_hz")

    ax.plot(
        sub["flow_rate_hz"],
        sub[PCK_COL],
        marker="o",
        linewidth=2,
        label=f"detection {1 / net_period:g} Hz ({net_period:g} s)",
    )

ax.set_xlabel("Optical-flow rate (Hz)")
ax.set_ylabel(f"Normalized PCK@{PCK_THRESHOLD:.1f} (%)")
ax.set_title(f"Experiment B: normalized PCK@{PCK_THRESHOLD:.1f} vs optical-flow rate")
ax.grid(True, alpha=0.3)
ax.legend()

if YLIM_PCK04 is not None:
    ax.set_ylim(YLIM_PCK04)



savefig(f"normalized_pck_{PCK_THRESHOLD:.1f}_vs_flow_rate_zoom.png")
plt.show()

## 9. Diminishing returns and text summary


In [ ]:
pck_col = f"pck_{PRIMARY_PCK_THRESHOLD:.1f}"

return_rows = []
for net_period, sub in summary.groupby("network_period_s"):
    sub = sub.sort_values("flow_rate_hz")
    baseline = sub.iloc[0]
    best_mpjpe = sub["mpjpe_px"].min()
    best_pck = sub[pck_col].max()

    for _, r in sub.iterrows():
        extra_hz = r["flow_rate_hz"] - baseline["flow_rate_hz"]
        mpjpe_gain = baseline["mpjpe_px"] - r["mpjpe_px"]
        pck_gain = r[pck_col] - baseline[pck_col]
        return_rows.append({
            "network_period_s": net_period,
            "detection_rate_hz": r["detection_rate_hz"],
            "flow_period_s": r["flow_period_s"],
            "flow_rate_hz": r["flow_rate_hz"],
            "mpjpe_px": r["mpjpe_px"],
            pck_col: r[pck_col],
            "mpjpe_gain_vs_slowest_flow_px": mpjpe_gain,
            "pck_gain_vs_slowest_flow_pctpt": pck_gain,
            "extra_flow_rate_vs_slowest_hz": extra_hz,
            "mpjpe_gain_px_per_extra_flow_hz": mpjpe_gain / extra_hz if extra_hz > 0 else np.nan,
            "pck_gain_pctpt_per_extra_flow_hz": pck_gain / extra_hz if extra_hz > 0 else np.nan,
            "mpjpe_gap_to_best_px": r["mpjpe_px"] - best_mpjpe,
            "pck_gap_to_best_pctpt": best_pck - r[pck_col],
        })

returns = pd.DataFrame(return_rows)
display(returns)

returns_path = ANALYSIS_OUT_DIR / "experimentB_diminishing_returns.csv"
returns.to_csv(returns_path, index=False)
print("Saved:", returns_path)

MPJPE_PLATEAU_TOLERANCE_PX = 1.0
PCK_PLATEAU_TOLERANCE_PCTPT = 1.0

plateau_rows = []
for net_period, sub in summary.groupby("network_period_s"):
    sub = sub.sort_values("flow_rate_hz")
    best_mpjpe = sub["mpjpe_px"].min()
    best_pck = sub[pck_col].max()

    candidates = sub[
        (sub["mpjpe_px"] <= best_mpjpe + MPJPE_PLATEAU_TOLERANCE_PX) &
        (sub[pck_col] >= best_pck - PCK_PLATEAU_TOLERANCE_PCTPT)
    ].sort_values("flow_rate_hz")

    if len(candidates):
        c = candidates.iloc[0]
        plateau_rows.append({
            "network_period_s": net_period,
            "detection_rate_hz": c["detection_rate_hz"],
            "recommended_flow_period_s": c["flow_period_s"],
            "recommended_flow_rate_hz": c["flow_rate_hz"],
            "mpjpe_px": c["mpjpe_px"],
            pck_col: c[pck_col],
        })

plateau = pd.DataFrame(plateau_rows)
display(plateau)

plateau_path = ANALYSIS_OUT_DIR / "experimentB_plateau_candidates.csv"
plateau.to_csv(plateau_path, index=False)
print("Saved:", plateau_path)

def fmt(x, digits=3):
    if pd.isna(x):
        return "n/a"
    return f"{x:.{digits}f}"

best_mpjpe = summary.loc[summary["mpjpe_px"].idxmin()]
best_pck = summary.loc[summary[pck_col].idxmax()]

lines = []
lines.append("Best configuration by MPJPE:")
lines.append(
    f"  network_period={best_mpjpe['network_period_s']:g} s "
    f"({best_mpjpe['detection_rate_hz']:g} Hz), "
    f"flow_period={best_mpjpe['flow_period_s']:g} s "
    f"({best_mpjpe['flow_rate_hz']:g} Hz), "
    f"MPJPE={fmt(best_mpjpe['mpjpe_px'], 3)} px, "
    f"normalized PCK@{PRIMARY_PCK_THRESHOLD:.1f}={fmt(best_mpjpe[pck_col], 2)}%."
)

lines.append("")
lines.append("Best configuration by normalized PCK:")
lines.append(
    f"  network_period={best_pck['network_period_s']:g} s "
    f"({best_pck['detection_rate_hz']:g} Hz), "
    f"flow_period={best_pck['flow_period_s']:g} s "
    f"({best_pck['flow_rate_hz']:g} Hz), "
    f"MPJPE={fmt(best_pck['mpjpe_px'], 3)} px, "
    f"normalized PCK@{PRIMARY_PCK_THRESHOLD:.1f}={fmt(best_pck[pck_col], 2)}%."
)

if len(plateau):
    lines.append("")
    lines.append("Low-cost plateau candidates:")
    for _, r in plateau.iterrows():
        lines.append(
            f"  detection {r['detection_rate_hz']:g} Hz: "
            f"flow {r['recommended_flow_rate_hz']:g} Hz "
            f"({r['recommended_flow_period_s']:g} s), "
            f"MPJPE={fmt(r['mpjpe_px'], 3)} px, "
            f"normalized PCK@{PRIMARY_PCK_THRESHOLD:.1f}={fmt(r[pck_col], 2)}%."
        )

text = "\n".join(lines)
print(text)

text_path = ANALYSIS_OUT_DIR / "experimentB_interpretation.txt"
text_path.write_text(text, encoding="utf-8")
print("\nSaved:", text_path)
